# MP-SENet — Finetune tiếp cho đủ 20 epoch (resume từ checkpoint cũ) + Infer trên tập TEST

Notebook này chạy tuần tự, chỉ cần bấm **Run All**:
1. Clone repo + cài thư viện
2. Tạo danh sách file train/validation (khớp tên CLEAN ↔ NOISE)
3. Patch `train.py` để hỗ trợ warm-start từ checkpoint chỉ có generator (`g_best_dns`)
4. **Khôi phục checkpoint cũ** từ Kaggle Dataset `Checkpoint MP-SEnet` (bị Kaggle tự giải nén hỏng định dạng khi upload) và sao chép vào thư mục làm việc
5. Tự phát hiện: đã hoàn thành bao nhiêu epoch → chỉ finetune tiếp phần còn thiếu để đạt **tổng cộng 20 epoch**; nếu chưa có checkpoint nào → warm-start từ `g_best_dns` và train đủ 20 epoch từ đầu
6. Lấy `g_best` (checkpoint PESQ tốt nhất) sau khi finetune xong, chạy inference trên toàn bộ thư mục `TEST`
7. Nén checkpoint + kết quả infer để tải về (checkpoint dùng định dạng `.tar.gz` để tránh bị Kaggle tự giải nén hỏng lần nữa)

> **Lưu ý quan trọng về checkpoint cũ:** dataset `Checkpoint MP-SEnet` chỉ có file `g_xxxxxxxx`
> (trọng số generator), không có `do_xxxxxxxx` (optimizer + discriminator + số step/epoch).
> Vì vậy notebook **không resume chính xác 100%** (không khôi phục được optimizer state) —
> nó sẽ warm-start lại từ file `g_` mới nhất và chạy tiếp số epoch còn thiếu để đạt tổng 20 epoch.


In [1]:
# ==== BƯỚC 1: Clone repo + cài thư viện ====
import os, subprocess

# Lớp phòng thủ thêm: nếu sau này có ai bật lại multi-GPU (FORCE_SINGLE_GPU=False ở Bước 2),
# các biến này giúp giảm khả năng NCCL bị treo trên hạ tầng Kaggle (không ảnh hưởng gì khi chỉ dùng 1 GPU).
os.environ.setdefault("NCCL_P2P_DISABLE", "1")
os.environ.setdefault("NCCL_SHM_DISABLE", "1")
os.environ.setdefault("NCCL_IB_DISABLE", "1")
# Giam phan manh bo nho CUDA (giup tan dung VRAM tot hon, do OOM do fragmentation)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_DIR = "/kaggle/working/MP-SENet"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/hkha0801-sketch/MP-SENet.git", REPO_DIR], check=True)
else:
    print("Repo đã tồn tại, bỏ qua clone.")

subprocess.run(["pip", "install", "-q", "librosa", "soundfile", "einops", "joblib", "natsort", "pesq", "rich"], check=True)
print("Cài đặt xong.")

Cloning into '/kaggle/working/MP-SENet'...
Updating files: 100% (250/250), done.


Cài đặt xong.


In [2]:
# ==== BƯỚC 2: Cấu hình đường dẫn & tham số (chỉnh ở đây nếu cần) ====
import os

# ---- FIX: treo cứng "Epoch: 1" (không lỗi, không log) trên Kaggle 2xT4/2xP100 ----
# Nguyên nhân: torch.cuda.device_count() = 2 trên Kaggle -> train.py TỰ ĐỘNG bật
# DistributedDataParallel (NCCL). Container Kaggle không hỗ trợ đầy đủ P2P/shared-memory
# mà NCCL cần để đồng bộ 2 GPU -> init_process_group()/all-reduce đầu tiên bị treo VĨNH VIỄN,
# hoàn toàn im lặng (đúng triệu chứng: in "Epoch: 1" rồi đứng im, không traceback).
# Cách chắc ăn nhất: ép chỉ dùng 1 GPU, bỏ qua DDP/NCCL hoàn toàn.
# batch_size trong config chỉ là 4 nên 1 GPU T4 vẫn đủ nhanh cho việc finetune.
# Nếu sau này muốn thử lại 2 GPU, đổi FORCE_SINGLE_GPU = False (không đảm bảo hết treo).
FORCE_SINGLE_GPU = True
if FORCE_SINGLE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("Đã ép chỉ dùng 1 GPU (CUDA_VISIBLE_DEVICES=0) để tránh treo NCCL trên Kaggle.")

# --- Đường dẫn dữ liệu (input, read-only) ---
BASE_INPUT  = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH"
TRAIN_CLEAN = os.path.join(BASE_INPUT, "TRAIN", "CLEAN")
TRAIN_NOISE = os.path.join(BASE_INPUT, "TRAIN", "NOISE")
TEST_DIR    = os.path.join(BASE_INPUT, "TEST")

# --- Checkpoint đã finetune một phần từ lần chạy trước (Kaggle Dataset, read-only) ---
# LƯU Ý: khi upload 1 file .zip lên làm Kaggle Dataset, Kaggle TỰ ĐỘNG giải nén nó.
# Mỗi checkpoint của PyTorch (g_xxxxxxxx) BẢN THÂN NÓ cũng là 1 file zip (định dạng lưu mặc
# định của torch.save) -> Kaggle giải nén luôn cả những file này -> 1 FILE duy nhất biến thành
# 1 THƯ MỤC chứa các file rời (data.pkl, byteorder, version, ...) -> torch.load() không đọc
# được nữa. BƯỚC 4c dưới đây sẽ tự "đóng gói" (rezip) lại các checkpoint này, không cần chỉnh
# sửa gì trên dataset gốc.
OLD_CKPT_INPUT_DIR = "/kaggle/input/datasets/foxduck/checkpoint-mp-senet/checkpoint"

# --- Đường dẫn làm việc (output, ghi được) ---
FILELIST_DIR      = "/kaggle/working/filelists"
CKPT_DIR          = "/kaggle/working/cp_finetune_dns"
CONFIG_FILE       = os.path.join(REPO_DIR, "best_ckpt", "config.json")
WARM_START_CKPT   = os.path.join(REPO_DIR, "best_ckpt", "g_best_dns")
INFER_OUTPUT_DIR  = "/kaggle/working/generated_files/finetuned_test_output"

# --- Tham số finetune ---
TARGET_TOTAL_EPOCHS = 20   # TỔNG số epoch muốn đạt được (không phải số epoch cộng thêm mỗi lần chạy)
VAL_RATIO            = 0.05  # tỉ lệ trích validation từ tập train

print("Cấu hình đã sẵn sàng.")


Đã ép chỉ dùng 1 GPU (CUDA_VISIBLE_DEVICES=0) để tránh treo NCCL trên Kaggle.
Cấu hình đã sẵn sàng.


In [3]:
# ==== BƯỚC 3: Tạo danh sách file train/validation (chỉ tạo 1 lần, giữ nguyên khi resume) ====
import random

os.makedirs(FILELIST_DIR, exist_ok=True)
train_list_path = os.path.join(FILELIST_DIR, "training.txt")
val_list_path   = os.path.join(FILELIST_DIR, "test.txt")

if os.path.exists(train_list_path) and os.path.exists(val_list_path):
    print("Danh sách file train/val đã tồn tại, giữ nguyên (đảm bảo resume nhất quán).")
else:
    clean_names = {os.path.splitext(f)[0] for f in os.listdir(TRAIN_CLEAN) if f.lower().endswith('.wav')}
    noisy_names = {os.path.splitext(f)[0] for f in os.listdir(TRAIN_NOISE) if f.lower().endswith('.wav')}
    common = sorted(clean_names & noisy_names)

    missing_clean = noisy_names - clean_names
    missing_noise = clean_names - noisy_names
    print(f"Clean: {len(clean_names)} | Noise: {len(noisy_names)} | Khớp cả 2: {len(common)}")
    if missing_clean:
        print(f"⚠️ {len(missing_clean)} file trong NOISE không có bản CLEAN tương ứng (sẽ bị bỏ qua).")
    if missing_noise:
        print(f"⚠️ {len(missing_noise)} file trong CLEAN không có bản NOISE tương ứng (sẽ bị bỏ qua).")
    assert len(common) > 0, "Không tìm thấy cặp file CLEAN/NOISE nào khớp tên nhau!"

    random.seed(1234)
    random.shuffle(common)
    val_size  = max(1, int(len(common) * VAL_RATIO))
    val_set   = common[:val_size]
    train_set = common[val_size:]

    with open(train_list_path, "w") as f:
        f.write("\n".join(train_set))
    with open(val_list_path, "w") as f:
        f.write("\n".join(val_set))

    print(f"Train: {len(train_set)} | Val: {len(val_set)}")

Clean: 8460 | Noise: 8460 | Khớp cả 2: 8460
Train: 8037 | Val: 423


In [4]:
# ==== BƯỚC 4: Patch train.py để hỗ trợ warm-start từ checkpoint chỉ có generator (idempotent) ====
train_py_path = os.path.join(REPO_DIR, "train.py")
with open(train_py_path, "r") as f:
    content = f.read()

if "warm_start_checkpoint" in content:
    print("train.py đã được patch từ trước, bỏ qua.")
else:
    old_block = """    steps = 0
    if cp_g is None or cp_do is None:
        state_dict_do = None
        last_epoch = -1
    else:"""
    new_block = """    steps = 0
    if cp_g is None or cp_do is None:
        state_dict_do = None
        last_epoch = -1
        if a.warm_start_checkpoint is not None:
            print(\"Warm-starting generator from '{}'\".format(a.warm_start_checkpoint))
            state_dict_g = load_checkpoint(a.warm_start_checkpoint, device)
            generator.load_state_dict(state_dict_g['generator'])
    else:"""
    assert old_block in content, "Không tìm thấy đoạn code cần patch — có thể repo đã thay đổi cấu trúc."
    content = content.replace(old_block, new_block)

    old_arg = "    parser.add_argument('--best_checkpoint_start_epoch', default=40, type=int)"
    new_arg = old_arg + "\n    parser.add_argument('--warm_start_checkpoint', default=None)"
    assert old_arg in content
    content = content.replace(old_arg, new_arg)

    with open(train_py_path, "w") as f:
        f.write(content)
    print("Đã patch train.py thành công.")

Đã patch train.py thành công.


In [5]:
# ==== BƯỚC 4b: Patch discriminator.py — fix treo (deadlock) do joblib fork/spawn lồng trong tiến trình CUDA/DDP ====
# batch_pesq() gốc dùng Parallel(n_jobs=15) (backend mặc định 'loky' -> spawn tiến trình con).
# Spawn tiến trình con bên trong 1 tiến trình đã init CUDA (do torch DDP mp.spawn tạo ra) rất dễ
# gây deadlock im lặng trên Kaggle (do /dev/shm hạn chế) -> training treo cứng ngay batch đầu tiên,
# không có traceback, không có log lỗi (đúng như log bạn thấy: in "Epoch: 1" xong rồi đứng im).
# Fix: chuyển sang backend 'threading' (không fork tiến trình, an toàn với CUDA) + giảm n_jobs cho hợp lý.

disc_py_path = os.path.join(REPO_DIR, "models", "discriminator.py")
with open(disc_py_path, "r") as f:
    disc_content = f.read()

old_line = 'pesq_score = Parallel(n_jobs=15)(delayed(cal_pesq)(c, n) for c, n in zip(clean, noisy))'
new_line = 'pesq_score = Parallel(n_jobs=min(len(clean), 4), backend="threading")(delayed(cal_pesq)(c, n) for c, n in zip(clean, noisy))'

if old_line not in disc_content:
    if new_line in disc_content:
        print("discriminator.py đã được patch từ trước, bỏ qua.")
    else:
        raise AssertionError("Không tìm thấy dòng cần patch trong discriminator.py — có thể repo đã thay đổi cấu trúc.")
else:
    disc_content = disc_content.replace(old_line, new_line)
    with open(disc_py_path, "w") as f:
        f.write(disc_content)
    print("Đã patch discriminator.py thành công (fix treo do joblib).")


Đã patch discriminator.py thành công (fix treo do joblib).


In [6]:
# ==== BƯỚC 4c: Khôi phục checkpoint cũ (bị Kaggle giải nén hỏng) + sao chép vào thư mục làm việc ====
# Xem giải thích lỗi ở BƯỚC 2. Hàm dưới đây phát hiện checkpoint nào đã bị Kaggle giải nén
# thành thư mục (thay vì là 1 file), rồi "đóng gói" (zip) lại đúng cấu trúc nội bộ mà
# torch.load() cần (thư mục gốc bên trong zip trùng tên file checkpoint, chứa data.pkl,
# byteorder, version, ...). Nếu checkpoint vẫn còn nguyên là 1 file bình thường thì chỉ copy.
import shutil, re, zipfile

os.makedirs(CKPT_DIR, exist_ok=True)

def _find_content_root(folder):
    """Tìm thư mục con chứa data.pkl — đó là gốc thật của checkpoint đã bị giải nén."""
    for root, dirs, files in os.walk(folder):
        if "data.pkl" in files:
            return root
    return None

def _repair_checkpoint_to_file(src_path, dest_path, ckpt_name):
    """Biến 1 checkpoint (dù đang là thư mục bị giải nén hay vẫn là file) thành 1 file
    duy nhất tại dest_path mà torch.load() đọc được."""
    if os.path.isfile(src_path):
        if os.path.abspath(src_path) != os.path.abspath(dest_path):
            shutil.copyfile(src_path, dest_path)
        return
    content_root = _find_content_root(src_path)
    if content_root is None:
        raise RuntimeError(
            f"Không tìm thấy data.pkl bên trong {src_path} — checkpoint có thể đã hỏng thật sự "
            f"(không chỉ do Kaggle giải nén)."
        )
    with zipfile.ZipFile(dest_path, "w", zipfile.ZIP_STORED) as zf:
        for root, dirs, files in os.walk(content_root):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, content_root)
                zf.write(full, os.path.join(ckpt_name, rel))

if os.path.isdir(OLD_CKPT_INPUT_DIR):
    pattern = re.compile(r"^(g|do)_\d{8}$")
    old_entries = sorted(e for e in os.listdir(OLD_CKPT_INPUT_DIR) if pattern.match(e))
    copied = []
    for name in old_entries:
        dest = os.path.join(CKPT_DIR, name)
        if os.path.exists(dest):
            # Đã có sẵn trong phiên làm việc hiện tại (tiến độ mới hơn, hoặc đã khôi phục
            # từ lần chạy trước trong cùng session) -> không ghi đè.
            continue
        src = os.path.join(OLD_CKPT_INPUT_DIR, name)
        _repair_checkpoint_to_file(src, dest, name)
        copied.append(name)
    if copied:
        print(f"Đã khôi phục & sao chép {len(copied)} checkpoint cũ vào {CKPT_DIR}:")
        for n in copied:
            print(f"  - {n}")
    else:
        print("Không có checkpoint mới nào cần khôi phục từ dataset cũ (hoặc đã tồn tại sẵn trong phiên này).")
else:
    print(f"Không tìm thấy thư mục checkpoint cũ ({OLD_CKPT_INPUT_DIR}) — bỏ qua, sẽ warm-start từ g_best_dns.")


Đã khôi phục & sao chép 32 checkpoint cũ vào /kaggle/working/cp_finetune_dns:
  - do_00004018
  - do_00008036
  - do_00012054
  - do_00016072
  - do_00020090
  - do_00024108
  - do_00028126
  - do_00032144
  - do_00036162
  - do_00040180
  - do_00044198
  - do_00048216
  - do_00052234
  - do_00056252
  - do_00060270
  - do_00064288
  - g_00004018
  - g_00008036
  - g_00012054
  - g_00016072
  - g_00020090
  - g_00024108
  - g_00028126
  - g_00032144
  - g_00036162
  - g_00040180
  - g_00044198
  - g_00048216
  - g_00052234
  - g_00056252
  - g_00060270
  - g_00064288


In [7]:
# ==== BƯỚC 5: Tự phát hiện trạng thái checkpoint và tính số epoch cần chạy tiếp ====
import glob, json, math, re
import torch

os.makedirs(CKPT_DIR, exist_ok=True)

with open(CONFIG_FILE) as f:
    cfg = json.load(f)

# ---- FIX: OOM khi ép 1 GPU (Bước 2) ----
# Trước đây với 2 GPU, train.py tự chia batch_size/num_gpus -> mỗi GPU chỉ xử lý batch_size/2.
# Sau khi ép chạy 1 GPU (FORCE_SINGLE_GPU=True) để tránh treo NCCL, GPU đó phải ôm nguyên batch_size
# gốc -> gấp đôi bộ nhớ cần dùng -> CUDA OutOfMemoryError.
# Cách sửa: khi chạy 1 GPU, tự giảm batch_size xuống một nửa (tối thiểu 1) để bù lại, và ghi ra
# 1 file config RIÊNG cho việc training (không đụng tới config gốc, vì config gốc còn dùng để infer
# và lưu trữ kiến trúc model, không liên quan tới batch_size).
TRAIN_CONFIG_FILE = os.path.join(CKPT_DIR, "config_train.json")
if globals().get("FORCE_SINGLE_GPU", False) and torch.cuda.device_count() <= 1:
    orig_batch_size = cfg["batch_size"]
    cfg["batch_size"] = max(1, orig_batch_size // 2)
    print(f"Chạy 1 GPU -> giảm batch_size: {orig_batch_size} -> {cfg['batch_size']} (tránh OOM).")
with open(TRAIN_CONFIG_FILE, "w") as f:
    json.dump(cfg, f, indent=4)
batch_size = cfg["batch_size"]

with open(train_list_path) as f:
    n_train = len([l for l in f if l.strip()])
steps_per_epoch = max(1, n_train // batch_size)

do_files = sorted(glob.glob(os.path.join(CKPT_DIR, "do_????????")))
g_files  = sorted(glob.glob(os.path.join(CKPT_DIR, "g_????????")))
warm_start_arg = None

if do_files:
    # Có đủ cặp g_/do_ -> train.py tự resume đầy đủ (kể cả optimizer, epoch, step).
    latest_do = do_files[-1]
    ckpt = torch.load(latest_do, map_location="cpu")
    current_epoch = ckpt["epoch"]
    current_step  = ckpt["steps"]
    target_epochs = max(current_epoch + 1, TARGET_TOTAL_EPOCHS)
    print(f"Tìm thấy checkpoint cũ (đầy đủ, có do_): {latest_do}")
    print(f"Epoch hiện tại: {current_epoch} | Step hiện tại: {current_step}")
    print(f"=> Sẽ resume và finetune tiếp tới epoch {target_epochs} (mục tiêu tổng {TARGET_TOTAL_EPOCHS} epoch).")
elif g_files:
    # Chỉ có generator (g_), KHÔNG có do_ (thiếu optimizer/discriminator/step state)
    # -> không resume đúng nghĩa được, chỉ warm-start lại từ trọng số g_ mới nhất.
    latest_g = g_files[-1]
    latest_step = int(re.search(r"(\d{8})$", latest_g).group(1))
    completed_epochs = latest_step // steps_per_epoch
    target_epochs = max(1, TARGET_TOTAL_EPOCHS - completed_epochs)
    warm_start_arg = latest_g
    print(f"Tìm thấy checkpoint generator cũ: {latest_g}")
    print("(Chỉ có g_, thiếu do_ nên không resume được optimizer/step -> sẽ warm-start lại từ đây.)")
    print(f"Ước tính đã hoàn thành ~{completed_epochs} epoch (step {latest_step} / {steps_per_epoch} step mỗi epoch).")
    print(f"=> Sẽ warm-start và finetune tiếp {target_epochs} epoch nữa để đạt tổng {TARGET_TOTAL_EPOCHS} epoch.")
else:
    target_epochs = TARGET_TOTAL_EPOCHS
    warm_start_arg = WARM_START_CKPT
    print("Chưa có checkpoint finetune nào — sẽ warm-start từ g_best_dns.")
    print(f"=> Sẽ finetune đủ {TARGET_TOTAL_EPOCHS} epoch từ đầu.")

checkpoint_interval = steps_per_epoch  # ~1 checkpoint/epoch
print(f"Steps/epoch: {steps_per_epoch} | checkpoint_interval = validation_interval = {checkpoint_interval}")


Chạy 1 GPU -> giảm batch_size: 4 -> 2 (tránh OOM).
Tìm thấy checkpoint cũ (đầy đủ, có do_): /kaggle/working/cp_finetune_dns/do_00064288
Epoch hiện tại: 15 | Step hiện tại: 64288
=> Sẽ resume và finetune tiếp tới epoch 20 (mục tiêu tổng 20 epoch).
Steps/epoch: 4018 | checkpoint_interval = validation_interval = 4018


In [8]:
# ==== BƯỚC 6: Chạy finetune ====
cmd = [
    "python", "train.py",
    "--config", TRAIN_CONFIG_FILE,
    "--input_clean_wavs_dir", TRAIN_CLEAN,
    "--input_noisy_wavs_dir", TRAIN_NOISE,
    "--input_training_file", train_list_path,
    "--input_validation_file", val_list_path,
    "--checkpoint_path", CKPT_DIR,
    "--training_epochs", str(target_epochs),
    "--stdout_interval", "20",
    "--checkpoint_interval", str(checkpoint_interval),
    "--validation_interval", str(checkpoint_interval),
    "--best_checkpoint_start_epoch", "1",
]
if warm_start_arg is not None:
    cmd += ["--warm_start_checkpoint", warm_start_arg]

print("Lệnh chạy:", " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_DIR)
result.check_returncode()
print("Finetune hoàn tất.")

Lệnh chạy: python train.py --config /kaggle/working/cp_finetune_dns/config_train.json --input_clean_wavs_dir /kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN --input_noisy_wavs_dir /kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE --input_training_file /kaggle/working/filelists/training.txt --input_validation_file /kaggle/working/filelists/test.txt --checkpoint_path /kaggle/working/cp_finetune_dns --training_epochs 20 --stdout_interval 20 --checkpoint_interval 4018 --validation_interval 4018 --best_checkpoint_start_epoch 1
Initializing Training Process..
Batch size per GPU : 2
MPNet(
  (dense_encoder): DenseEncoder(
    (dense_conv_1): Sequential(
      (0): Conv2d(2, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): PReLU(num_parameters=64)
    )
    (dense_block): DenseBlock(
      (dense_block): ModuleList(
        (0): Sequential(
          (

In [9]:
# ==== BƯỚC 7: Lấy best checkpoint (theo PESQ) ====
best_ckpt_path = os.path.join(CKPT_DIR, "g_best")
assert os.path.isfile(best_ckpt_path), f"Không tìm thấy {best_ckpt_path} — kiểm tra log training ở trên (có thể chưa đủ epoch để vượt best_checkpoint_start_epoch)."

best_config_path = os.path.join(CKPT_DIR, "config.json")
if not os.path.isfile(best_config_path):
    import shutil
    shutil.copyfile(CONFIG_FILE, best_config_path)

print(f"Best checkpoint: {best_ckpt_path}")

Best checkpoint: /kaggle/working/cp_finetune_dns/g_best


In [10]:
# ==== BƯỚC 8: Infer trên tập TEST bằng best checkpoint vừa finetune ====
os.makedirs(INFER_OUTPUT_DIR, exist_ok=True)

cmd = [
    "python", "inference.py",
    "--checkpoint_file", best_ckpt_path,
    "--input_noisy_wavs_dir", TEST_DIR,
    "--output_dir", INFER_OUTPUT_DIR,
]
print("Lệnh chạy:", " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_DIR)
result.check_returncode()
print(f"Infer xong. Kết quả tại: {INFER_OUTPUT_DIR}")

Lệnh chạy: python inference.py --checkpoint_file /kaggle/working/cp_finetune_dns/g_best --input_noisy_wavs_dir /kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST --output_dir /kaggle/working/generated_files/finetuned_test_output
Initializing Inference Process..
Loading '/kaggle/working/cp_finetune_dns/g_best'
Complete.


Traceback (most recent call last):
  File "/kaggle/working/MP-SENet/inference.py", line 91, in <module>
    main()
  File "/kaggle/working/MP-SENet/inference.py", line 87, in main
    inference(a)
  File "/kaggle/working/MP-SENet/inference.py", line 40, in inference
    test_indexes = os.listdir(a.input_noisy_wavs_dir)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST'


CalledProcessError: Command '['python', 'inference.py', '--checkpoint_file', '/kaggle/working/cp_finetune_dns/g_best', '--input_noisy_wavs_dir', '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST', '--output_dir', '/kaggle/working/generated_files/finetuned_test_output']' returned non-zero exit status 1.

In [ ]:
# ==== BƯỚC 9: Nén checkpoint + kết quả infer để tải về ====
import shutil

zip_ckpt_base  = "/kaggle/working/cp_finetune_dns"
zip_infer_base = "/kaggle/working/finetuned_test_output"

# Checkpoint: dùng .tar.gz thay vì .zip.
# LÝ DO: mỗi file checkpoint (g_xxxxxxxx, do_xxxxxxxx) BẢN THÂN NÓ đã là 1 file zip
# (định dạng lưu mặc định của torch.save). Nếu nén cả thư mục checkpoint bằng .zip rồi
# upload lại làm Kaggle Dataset, Kaggle sẽ tự giải nén file .zip đó — và giải nén luôn
# cả các file checkpoint bên trong (vì chúng cũng có định dạng zip), biến 1 file thành
# 1 thư mục rời rạc, khiến torch.load() không đọc được nữa (đây chính là lỗi bạn gặp
# với dataset "Checkpoint MP-SEnet"). .tar.gz không bị Kaggle tự giải nén nên an toàn hơn.
shutil.make_archive(zip_ckpt_base, "gztar", CKPT_DIR)
shutil.make_archive(zip_infer_base, "zip", INFER_OUTPUT_DIR)

print("Đã tạo file nén (vào tab Output của Kaggle để tải):")
print(f" - {zip_ckpt_base}.tar.gz   (toàn bộ checkpoint finetune — dùng file này để upload làm Dataset checkpoint cho lần chạy sau)")
print(f" - {zip_infer_base}.zip     (kết quả infer trên tập TEST)")
